In [1]:
!pip install -q \
transformers \
accelerate \
openpyxl \
scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [4]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

In [5]:
print(df.columns)

Index(['id', 'comment', 'label'], dtype='object')


In [6]:
df = df[['comment', 'label']]

In [7]:
df.dropna(inplace=True)

In [8]:
df['label'] = df['label'].astype(int)

In [9]:
print(df.head())

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0
label
0    1577
1    1471
Name: count, dtype: int64


In [10]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [11]:
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [13]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [14]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

In [15]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [16]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    warmup_ratio=0.1,

    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    save_total_limit=1
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.653283,0.684343,0.568852,0.545239,0.602517,0.577822
2,0.690533,0.683401,0.531148,0.458071,0.610847,0.545100
3,0.688629,0.672278,0.562295,0.502807,0.660399,0.575519
4,0.702601,0.676256,0.518033,0.341253,0.259016,0.500000
5,0.689181,0.682571,0.613115,0.612911,0.612948,0.613084
6,0.652005,0.661620,0.613115,0.612177,0.617665,0.615689
7,0.643444,0.659681,0.604918,0.600658,0.616129,0.609436
8,0.670840,0.653661,0.608197,0.604239,0.619090,0.612600
9,0.627656,0.659084,0.611475,0.607287,0.623097,0.616001
10,0.618062,0.664825,0.600000,0.593693,0.614457,0.605281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=3050, training_loss=0.6766046388032007, metrics={'train_runtime': 1489.6624, 'train_samples_per_second': 16.366, 'train_steps_per_second': 2.047, 'total_flos': 1603661882419200.0, 'train_loss': 0.6766046388032007, 'epoch': 10.0})

In [19]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.618062,0.682402,10,0.613115,0.612911,0.612948,0.613084


{'eval_loss': 0.6824023723602295, 'eval_accuracy': 0.6131147540983607, 'eval_f1': 0.6129108587162279, 'eval_precision': 0.6129483099681776, 'eval_recall': 0.6130844742960475}


In [20]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(-1)

cm = confusion_matrix(test_labels, preds)

print(cm)

[[194 122]
 [114 180]]


In [21]:
print(
    classification_report(
        test_labels,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.63      0.61      0.62       316
           1       0.60      0.61      0.60       294

    accuracy                           0.61       610
   macro avg       0.61      0.61      0.61       610
weighted avg       0.61      0.61      0.61       610



In [22]:
model.save_pretrained("./mbert_model")

tokenizer.save_pretrained("./mbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mbert_model/tokenizer_config.json', './mbert_model/tokenizer.json')

In [23]:
!zip -r sarcasm_model.zip sarcasm_model

	zip warning: name not matched: sarcasm_model

zip error: Nothing to do! (try: zip -r sarcasm_model.zip . -i sarcasm_model)


In [24]:
text = "අද නම් හරිම ලස්සන වැඩක් කරලා 😂"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

if prediction.item() == 1:
    print("Sarcastic")
else:
    print("Non-Sarcastic")

Prediction: 1
Sarcastic


In [25]:
from sklearn.metrics import classification_report

print(classification_report(test_labels, preds))

              precision    recall  f1-score   support

           0       0.63      0.61      0.62       316
           1       0.60      0.61      0.60       294

    accuracy                           0.61       610
   macro avg       0.61      0.61      0.61       610
weighted avg       0.61      0.61      0.61       610

